# 3. Model comparison

**Question.** Trained on 2011 to 2014, validated on 2015 to 2016 and tested on 2017 to 2018: does XGBoost
beat the logistic regression baseline, and by enough to matter? Random forest and a model using only
age and sex are included for reference.

Rules: hyperparameters and the final model choice use the validation cycle only. The test cycle is
scored once. Intervals are bootstrap 95% intervals over test rows.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from nhanes import nb, plots
from nhanes.features import CATEGORICAL, FEATURES, LABELS, NUMERIC, TARGET

ctx = nb.context()
df, R = ctx.df, ctx.R
pct = nb.pct
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 160)
print("mode:", "synthetic fixture (CI)" if ctx.fixture else "real NHANES data")
print(f"{len(df):,} adults, {int(df[TARGET].sum()):,} reporting heart disease")

mode: real NHANES data
22,512 adults, 1,637 reporting heart disease


## 3.1 Test-cycle scores, recomputed from the saved predictions

In [2]:
from sklearn.metrics import average_precision_score, roc_auc_score

pred = ctx.pred
test = pred[pred.split == "test"]
valid = pred[pred.split == "valid"]
models = ["age_sex_baseline", "logistic_regression", "random_forest", "xgboost"]
rows = []
for m in models:
    r = R["models"][m]["test"]
    rows.append(
        {
            "model": m,
            "valid PR-AUC": average_precision_score(valid[TARGET], valid[f"p_{m}"]),
            "test ROC-AUC": roc_auc_score(test[TARGET], test[f"p_{m}"]),
            "ROC-AUC 95% low": r["roc_auc_ci"][0],
            "ROC-AUC 95% high": r["roc_auc_ci"][1],
            "test PR-AUC": average_precision_score(test[TARGET], test[f"p_{m}"]),
            "PR-AUC 95% low": r["pr_auc_ci"][0],
            "PR-AUC 95% high": r["pr_auc_ci"][1],
            "Brier": r["brier"],
        }
    )
    nb.check(f"{m} test ROC-AUC", rows[-1]["test ROC-AUC"], r["roc_auc"])
    nb.check(f"{m} test PR-AUC", rows[-1]["test PR-AUC"], r["pr_auc"])
table = pd.DataFrame(rows).set_index("model")
display(table)
print(
    f"Share of test adults with the outcome: {pct(test[TARGET].mean())}. A model that says 'no' to everyone scores {pct(1 - test[TARGET].mean())} accuracy."
)

Check: age_sex_baseline test ROC-AUC: notebook 0.795209 vs results.json 0.795209 (matches)
Check: age_sex_baseline test PR-AUC: notebook 0.22662 vs results.json 0.22662 (matches)
Check: logistic_regression test ROC-AUC: notebook 0.828528 vs results.json 0.828528 (matches)
Check: logistic_regression test PR-AUC: notebook 0.275695 vs results.json 0.275695 (matches)
Check: random_forest test ROC-AUC: notebook 0.828687 vs results.json 0.828687 (matches)
Check: random_forest test PR-AUC: notebook 0.284129 vs results.json 0.284129 (matches)
Check: xgboost test ROC-AUC: notebook 0.829535 vs results.json 0.829535 (matches)
Check: xgboost test PR-AUC: notebook 0.277167 vs results.json 0.277167 (matches)


,valid PR-AUC,test ROC-AUC,ROC-AUC 95% low,ROC-AUC 95% high,test PR-AUC,PR-AUC 95% low,PR-AUC 95% high,Brier
model,,,,,,,,
age_sex_baseline,0.229,0.795,0.775,0.813,0.227,0.198,0.258,0.069
logistic_regression,0.288,0.829,0.811,0.845,0.276,0.243,0.315,0.066
random_forest,0.300,0.829,0.812,0.846,0.284,0.249,0.325,0.066
xgboost,0.300,0.830,0.812,0.846,0.277,0.243,0.315,0.067


Share of test adults with the outcome: 8.3%. A model that says 'no' to everyone scores 91.7% accuracy.


## 3.2 ROC and precision-recall curves

In [3]:
y = test[TARGET].to_numpy()
probs = {m: test[f"p_{m}"].to_numpy() for m in models}
nb.show(plots.plot_roc_pr(y, probs))

<Figure size 1265x528 with 2 Axes>

## 3.3 Confusion matrices at the recall-target threshold (set on validation, see notebook 4)

In [4]:
thr = {m: R["models"][m]["recall_target"]["threshold"] for m in models}
nb.show(plots.plot_confusions(y, probs, thr, ["logistic_regression", "random_forest", "xgboost"]))

<Figure size 1419x462 with 3 Axes>

## 3.4 Calibration: do predicted risks match observed rates?

In [5]:
nb.show(plots.plot_calibration(y, probs))
cal = pd.DataFrame({m: R["models"][m]["calibration_test"] for m in models}).T[
    ["brier", "ece", "intercept", "slope", "mean_predicted", "observed_rate"]
]
display(cal)

<Figure size 1265x506 with 2 Axes>

,brier,ece,intercept,slope,mean_predicted,observed_rate
age_sex_baseline,0.069,0.010,0.101,1.004,0.076,0.083
logistic_regression,0.066,0.011,-0.045,0.959,0.081,0.083
random_forest,0.066,0.006,0.059,1.008,0.080,0.083
xgboost,0.067,0.010,0.070,1.007,0.079,0.083


## 3.5 Paired differences between models (same test rows, paired bootstrap)

In [6]:
diff = pd.DataFrame(
    {
        k: {
            "ROC-AUC gap": v["roc_auc"][0],
            "ROC-AUC low": v["roc_auc"][1],
            "ROC-AUC high": v["roc_auc"][2],
            "PR-AUC gap": v["pr_auc"][0],
            "PR-AUC low": v["pr_auc"][1],
            "PR-AUC high": v["pr_auc"][2],
        }
        for k, v in R["paired_differences_test"].items()
    }
).T
display(diff)

,ROC-AUC gap,ROC-AUC low,ROC-AUC high,PR-AUC gap,PR-AUC low,PR-AUC high
xgboost_minus_logistic_regression,0.001,-0.004,0.006,0.001,-0.014,0.016
random_forest_minus_logistic_regression,0.000,-0.007,0.007,0.008,-0.014,0.028
logistic_regression_minus_age_sex_baseline,0.033,0.022,0.044,0.049,0.026,0.079
xgboost_minus_age_sex_baseline,0.034,0.023,0.045,0.051,0.027,0.079


## 3.6 How sensitive are the tree models to their settings?

In [7]:
for m in ["random_forest", "xgboost"]:
    g = pd.DataFrame(R["models"][m]["grid"]).sort_values("valid_pr_auc", ascending=False)
    print(
        m,
        "validation PR-AUC across",
        len(g),
        "settings: best",
        pct(g.valid_pr_auc.max()),
        "worst",
        pct(g.valid_pr_auc.min()),
    )
    display(g.head(4))

random_forest validation PR-AUC across 10 settings: best 30.0% worst 26.4%


,min_samples_leaf,max_features,valid_pr_auc
5,40,0.500,0.300
3,20,0.500,0.299
1,10,0.500,0.298
7,80,0.500,0.296


xgboost validation PR-AUC across 8 settings: best 30.0% worst 29.0%


,max_depth,min_child_weight,n_estimators,valid_pr_auc
2,2,1,151,0.300
3,2,5,150,0.298
7,4,5,79,0.297
6,4,1,79,0.297


## 3.7 Does the time-based split matter?

A random split mixes survey years, so train and test come from the same period. Compare the
logistic regression's AUC under a random 5-fold split over all four cycles with its AUC on the
later test cycle.


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from nhanes.features import make_preprocessor

C = R["models"]["logistic_regression"]["params"]["C"]
pipe = Pipeline(
    [
        ("pre", make_preprocessor("median_indicator")),
        ("clf", LogisticRegression(C=C, max_iter=5000)),
    ]
)
p_cv = cross_val_predict(
    pipe,
    df[FEATURES],
    df[TARGET],
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    method="predict_proba",
)[:, 1]
auc_random = roc_auc_score(df[TARGET], p_cv)
auc_time = R["models"]["logistic_regression"]["test"]["roc_auc"]
print(f"Logistic regression ROC-AUC, random 5-fold over all cycles: {pct(auc_random)}")
print(
    f"Logistic regression ROC-AUC, trained earlier and tested on the latest cycle: {pct(auc_time)}"
)

Logistic regression ROC-AUC, random 5-fold over all cycles: 84.1%
Logistic regression ROC-AUC, trained earlier and tested on the latest cycle: 82.9%


## What this means

In [9]:
lr, xg, rf, base = (
    R["models"][m]["test"]
    for m in ["logistic_regression", "xgboost", "random_forest", "age_sex_baseline"]
)
dx = R["paired_differences_test"]["xgboost_minus_logistic_regression"]["roc_auc"]
dr = R["paired_differences_test"]["random_forest_minus_logistic_regression"]["roc_auc"]
db = R["paired_differences_test"]["logistic_regression_minus_age_sex_baseline"]["roc_auc"]
verdict = "does not beat" if dx[1] <= 0 <= dx[2] else ("beats" if dx[0] > 0 else "is worse than")
nb.say(
    f"- XGBoost **{verdict}** logistic regression on the test cycle: ROC-AUC {pct(xg['roc_auc'])} against {pct(lr['roc_auc'])}, "
    f"paired gap {dx[0] * 100:+.1f} points (95% interval {dx[1] * 100:+.1f} to {dx[2] * 100:+.1f}). "
    f"For the random forest the gap is {dr[0] * 100:+.1f} points ({dr[1] * 100:+.1f} to {dr[2] * 100:+.1f}).\n"
    f"- Age and sex alone reach {pct(base['roc_auc'])}; logistic regression on all features differs from that by "
    f"{db[0] * 100:+.1f} points (paired 95% interval {db[1] * 100:+.1f} to {db[2] * 100:+.1f}).\n"
    f"- Calibration: logistic regression predicts a mean risk of {pct(R['models']['logistic_regression']['calibration_test']['mean_predicted'])} "
    f"against an observed {pct(R['models']['logistic_regression']['calibration_test']['observed_rate'])}.\n"
    f"- A random split gives {pct(auc_random)} against {pct(auc_time)} on the later cycle, "
    f"a difference of {(auc_random - auc_time) * 100:+.1f} points."
)

- XGBoost **does not beat** logistic regression on the test cycle: ROC-AUC 83.0% against 82.9%, paired gap +0.1 points (95% interval -0.4 to +0.6). For the random forest the gap is +0.0 points (-0.7 to +0.7).
- Age and sex alone reach 79.5%; logistic regression on all features differs from that by +3.3 points (paired 95% interval +2.2 to +4.4).
- Calibration: logistic regression predicts a mean risk of 8.1% against an observed 8.3%.
- A random split gives 84.1% against 82.9% on the later cycle, a difference of +1.2 points.

## What is weak

In [10]:
nb.say(
    f"- Only {int(test[TARGET].sum())} test cases: intervals are wide ({(R['models']['logistic_regression']['test']['roc_auc_ci'][1] - R['models']['logistic_regression']['test']['roc_auc_ci'][0]) * 100:.1f} points wide for logistic regression), "
    "so small gaps between models cannot be resolved.\n"
    "- The model-selection rule was written down after an exploratory run in which I had seen test scores (see DESIGN.md section 6), "
    "and the tuning grids were chosen by hand, so treat the choice as lightly tuned.\n"
    "- The features are few and tabular; on such data logistic regression is often hard to beat. "
    "That says nothing about XGBoost on richer data.\n"
    "- Intervals treat respondents as independent draws and ignore NHANES clustering, so they are a little too narrow."
)

- Only 457 test cases: intervals are wide (3.4 points wide for logistic regression), so small gaps between models cannot be resolved.
- The model-selection rule was written down after an exploratory run in which I had seen test scores (see DESIGN.md section 6), and the tuning grids were chosen by hand, so treat the choice as lightly tuned.
- The features are few and tabular; on such data logistic regression is often hard to beat. That says nothing about XGBoost on richer data.
- Intervals treat respondents as independent draws and ignore NHANES clustering, so they are a little too narrow.